In [0]:
from pyspark.sql.functions import monotonically_increasing_id
# from pyspark import pipelines as dp
import requests

In [0]:
%sql
SELECT DISTINCT geography_type
FROM digital_redlining.fixed_fcc.fact_geo;

In [0]:
df = spark.read.table('digital_redlining.fixed_fcc.fact_geo')

display(df)

In [0]:

df = spark.read.format("csv").option("header", "true").load("/Volumes/digital_redlining/census_demo/src_files/ACSDP5Y2024.DP05-Data.csv")


df = df.withColumnRenamed('Geography', 'geography_id')
df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!White', 'percent_white')
df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Black or African American!!African American', 'percent_black')
df = df.withColumnRenamed('Percent!!HISPANIC OR LATINO AND RACE!!Total population!!Hispanic or Latino (of any race)', 'percent_hispanic')
df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!American Indian and Alaska Native', 'percent_native')
df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Asian', 'percent_asian')
df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Native Hawaiian and Other Pacific Islander', 'percent_hawaiian')
df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Some Other Race', 'percent_other')

df = df.withColumn("id", monotonically_increasing_id())
df = df.select([
        'id', 
        'geography_id', 
        'percent_white', 
        'percent_black', 
        'percent_hispanic', 
        'percent_native', 
        'percent_asian', 
        'percent_hawaiian', 
        'percent_other'
    ])
    

In [0]:
def dim_race():

    df = spark.read.format("csv").option("header", "true").load("/Volumes/digital_redlining/census_demo/src_files/ACSDP5Y2024.DP05-Data.csv")

    df = df.withColumnRenamed('Geography', 'geography_id')
    df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!White', 'percent_white')
    df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Black or African American!!African American', 'percent_black')
    df = df.withColumnRenamed('Percent!!HISPANIC OR LATINO AND RACE!!Total population!!Hispanic or Latino (of any race)', 'percent_hispanic')
    df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!American Indian and Alaska Native', 'percent_native')
    df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Asian', 'percent_asian')
    df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Native Hawaiian and Other Pacific Islander', 'percent_hawaiian')
    df = df.withColumnRenamed('Percent!!RACE!!Total population!!One race!!Some Other Race', 'percent_other')

    df = df.withColumn("id", monotonically_increasing_id())
    df = df.select([
        'id', 
        'geography_id', 
        'percent_white', 
        'percent_black', 
        'percent_hispanic', 
        'percent_native', 
        'percent_asian', 
        'percent_hawaiian', 
        'percent_other'
    ])

    return df

In [0]:
print(dim_race().head())

In [0]:
# df = spark.read.table("digital_redlining.fixed_fcc.fact_geo")

In [0]:
# df = df.select('geography_type').dropDuplicates()

In [0]:
# display(df)

In [0]:
data = requests.get("https://api.census.gov/data/2024/acs/acs5/profile?get=DP05&ucgid=pseudo(0100000US$3100000)").json()


df_api = spark.createDataFrame(data)
display(df_api)

In [0]:
import requests, json

# 1️⃣ Get the variable dictionary
meta_url = "https://api.census.gov/data/2019/acs/acs1/variables.json"
meta = requests.get(meta_url).json()["variables"]

# 2️⃣ Build a simple lookup: code → label
lookup = {code: info["label"] for code, info in meta.items()}

# 3️⃣ Fetch the data you need
data_url = "https://api.census.gov/data/2023/acs/acs5/profile?get=group(DP05)&ucgid=pseudo(0100000US$3100000)"
rows = requests.get(data_url).json()

# 4️⃣ Replace the header row with plain English labels
header = rows[0]
plain_header = [lookup.get(col, col) for col in header]  # fallback to original if unknown
rows[0] = plain_header

# Wrap each row as a dict for schema inference
data_list = [dict(zip(plain_header, row)) for row in rows[1:]]

df_census = spark.createDataFrame(data_list)
display(df_census)

In [0]:
# https://api.census.gov/data/2024/acs/acs5/profile?get=group(DP05)&ucgid=pseudo(0100000US$3100000)

# 1️⃣ Get the variable dictionary
meta_url = "https://api.census.gov/data/2019/acs/acs1/variables.json"
meta = requests.get(meta_url).json()["variables"]

# 2️⃣ Build a simple lookup: code → label
lookup = {code: info["label"] for code, info in meta.items()}

# 3️⃣ Fetch the data you need
data_url = "https://api.census.gov/data/2024/acs/acs5/profile?get=group(DP05)&ucgid=pseudo(0100000US$3100000)"
rows = requests.get(data_url).json()

# 4️⃣ Replace the header row with plain English labels
header = rows[0]
plain_header = [lookup.get(col, col) for col in header]   # fallback to original if unknown
rows[0] = plain_header

# print(json.dumps(rows, indent=2))

df_census = spark.createDataFrame(rows[1:], schema=plain_header)
display(df_census)